# bert-1
- run pipeline

In [ ]:
from nlp import ClimateBERTAnalyzer, analyze_reports

stats = analyze_reports('../data/reports/Baosteel')
# stats = analyze_reports("../data/reports")

# bert-2
- run vizualisations

In [ ]:
from nlp import ClimateBERTVisualizer, visualize_results

visualize_results("../cache", "../out")

✅ Loaded: 15 reports, 1 companies, 2013-2020

EXPORTING CSV FILES
   ✓ company_year.csv (8 rows)
   ✓ company_totals.csv (1 companies)
   ✓ yearly_industry.csv (8 years)
   ✓ funnel_company_year.csv (8 rows)

   📁 All CSVs saved to: ../out/

GENERATING PLOTS
   ✓ slide_main.png
   ✓ slide_sentiment_trend.png
   ✓ talk_score_trend.png
   ✓ funnel_trend.png
   ✓ talk_score_per_company.png
   ✓ per_company_components.png
   ✓ per_company_sentiment.png
   ✓ sentiment_all_companies.png
   ✓ n0_funnel.png
   ✓ n0_quality_comparison.png
   ✓ n0_per_company.png
   ✓ n0_gap_analysis.png

   📁 All plots saved to: ../out/

   Generating word frequency plots...

   📊 ALL CHUNKS (top 30):
   environment(1232), iron(766), management(736), development(720), energy(628), technology(609), products(607), protection(559), production(525), industry(522), green(473), emission(453), system(428), reduction(421), base(410)

   🌱 OPPORTUNITY chunks (top 15):
   environment(809), development(574), technology(52

# rag 1

## Reload modules

run when changing code in .py file

In [2]:
%load_ext autoreload
%autoreload 2
# %aimport nlp.rag_1

# import os
# print(os.getcwd())

## Model Testing

Test which models work for extraction (format compliance + quality).

In [3]:
from nlp.rag_test import test_models, compare_extractions
from nlp import RAGConfig

MODELS = [
    # Groq (cloud API)
    RAGConfig(llm_provider="groq", model="llama-3.1-8b-instant", llm_num_ctx=128000),
    # RAGConfig(llm_provider="groq", model="llama-3.3-70b-versatile", llm_num_ctx=128000),
    # RAGConfig(llm_provider="groq", model="mixtral-8x7b-32768", llm_num_ctx=32768),
    # RAGConfig(llm_provider="groq", model="gemma2-9b-it", llm_num_ctx=8192),

    # Ollama (local)
    RAGConfig(llm_provider="ollama", model="qwen2.5:3b", llm_num_ctx=4096),  # 1.9GB
    # RAGConfig(llm_provider="ollama", model="qwen3:4b", llm_num_ctx=4096),  # 2.5GB, slow (thinking)
    # RAGConfig(llm_provider="ollama", model="qwen3:8b", llm_num_ctx=4096),  # 5.2GB
    # RAGConfig(llm_provider="ollama", model="gemma3:1b", llm_num_ctx=4096),  # 815MB
    # RAGConfig(llm_provider="ollama", model="gemma3:4b", llm_num_ctx=4096),  # 3.3GB
    # RAGConfig(llm_provider="ollama", model="llama3.2:3b", llm_num_ctx=4096),  # 2.0GB
    # RAGConfig(llm_provider="ollama", model="phi3:mini", llm_num_ctx=4096),  # 2.2GB, bad format
    # RAGConfig(llm_provider="ollama", model="llama3.1:8b", llm_num_ctx=4096),  # 4.9GB
]

results = test_models(MODELS, skip_extraction=False)
compare_extractions(results)

RAG MODEL TESTING (2 models) - 012/2021

--- groq/llama-3.1-8b-instant (ctx=128,000 → 296 chunks) ---
    Format: PASS (0.3s)
    Output: [012_001]|||The high cost of green hydrogen remains a significant barrier. Production costs are 4-6x higher than grey hydrogen.
[012_002]|||Infrastruc...
✓ RAG Pipeline initialized (GPU: NVIDIA T1200 Laptop GPU (3.9GB))
✓ Filtered 1048 chunks below detector_score=0.7 (14545 remaining)
('001', '2022'): 503 chunks
('001', '2023'): 487 chunks
('001', '2021'): 484 chunks
('015', '2024'): 344 chunks
('001', '2019'): 338 chunks
('001', '2024'): 336 chunks
('001', '2020'): 332 chunks
('003', '2024'): 279 chunks
('015', '2025'): 277 chunks
('001', '2013'): 262 chunks
✓ Loaded 14545 chunks from 15 companies (../cache)
Loading Groq: llama-3.1-8b-instant
    [DEBUG] Context preview (8599 chars):
    [012_005]
In addition, in order to improve the emissions, the extraction hood on the Coke Overhead Machine 1 (Investition 0.9 million €) was renewed and put into op

## Run Extraction

Configure and run the full extraction pipeline.

In [1]:
from nlp import load_pipeline, RAGConfig

config = RAGConfig(
    llm_provider="groq",
    model="llama-3.1-8b-instant",
    llm_num_ctx=128000,
)

pipeline = load_pipeline(config)
# pipeline.print_chunk_overview()


/home/as/code/ai/susteelaible/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ RAG Pipeline initialized (GPU: NVIDIA T1200 Laptop GPU (3.9GB))
✓ Loaded 15593 chunks from 15 companies (../cache)


In [ ]:
# Extract all companies (saves to out/)
results = pipeline.extract_all_companies()


In [ ]:
# Display results for a specific company
company_id = list(results.keys())[0]  # First company
df_barriers, df_motivators = results[company_id]
pipeline.display_results(company_id, df_barriers, df_motivators)

# rag 2

In [ ]:
# from bertopic.representation import OpenAI,LlamaCPP
from nlp import TopicModelConfig, run_topic_modeling_pipeline

# Set True to ignore cached model/embeddings and retrain from scratch
FORCE_RETRAIN = True

config = TopicModelConfig(
    # Embedding model
    # embedding_model="sentence-transformers/all-mpnet-base-v2",
    embedding_model="BAAI/bge-small-en-v1.5",

    batch_size=64,

    # UMAP (dimensionality reduction)
    umap_n_neighbors=30,
    umap_n_components=15,
    umap_min_dist=0.05,
    umap_metric='cosine',
    umap_random_state=42,

    # HDBSCAN (clustering)
    hdbscan_min_cluster_size=10,  # Higher = fewer topics
    hdbscan_min_samples=2,        # Lower = less outliers
    hdbscan_metric='euclidean',
    hdbscan_cluster_selection_method='leaf',  # 'eom' (inclusive) or 'leaf' (tight, more cleanup required)

    # Vectorizer (c-TF-IDF)
    vectorizer_ngram_range=(1, 2),
    vectorizer_min_df=1,
    vectorizer_max_df=0.95,

    # Topic representation
    mmr_diversity=0.2, # 0 - pure relevance, redundant/simi word ... 1 - pure diverse. max diff word
    top_n_words=10,
    nr_topics=10,  # Set None for auto, or int to reduce post-hoc
    calculate_probabilities=True,

    # Outlier reduction (post-hoc)
    reduce_outliers=False,  # Reassign outliers to nearest topic
    reduce_outliers_strategy='embeddings',  # 'embeddings', 'c-tf-idf', or 'distributions'

    # Visualization UMAP (separate 2D projection)
    viz_umap_n_neighbors=10,
    viz_umap_n_components=2,
    viz_umap_min_dist=0.0,

    # LLM for topic labeling
    ollama_model="gemma3:4b",  # Fast + follows format. Avoid qwen3 (hidden thinking = slow)
    ollama_base_url="http://localhost:11434",
    llm_temperature=0.0,

    # Misc
    verbose=True,
    embeddings_cache_path=None,
)

results = run_topic_modeling_pipeline(
    data_folder="../out",
    output_folder="../out/topics5",
    config=config,
    force_retrain=FORCE_RETRAIN
)
# Access results
barriers_df = results['barriers']['df']
motivators_df = results['motivators']['df']

In [ ]:
motivators_df